# Designing a FIR Filter

In [ ]:
import os
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt
from scipy.signal import lfilter, freqz
from rp.lia.config import calc_fir_coeffs, quantize_coeffs, quantize_and_dequantize_coeffs
from rp.lia.processing import calc_r_phi, reconstruct_signal

# Plotting settings
# plt.rcParams['axes.grid'] = True
plt.rcParams['grid.linestyle'] = '--'  # Grid line style
plt.rcParams['grid.linewidth'] = 0.5  # Grid line width
plt.rcParams['axes.titlesize'] = 14  # Title font size
plt.rcParams['axes.labelsize'] = 12  # Axis label font size
plt.rcParams['xtick.labelsize'] = 10  # X tick label font size
plt.rcParams['ytick.labelsize'] = 10  # Y tick label font size
plt.rcParams['legend.fontsize'] = 10  # Legend font size
plt.rcParams['xtick.direction'] = 'in'  # X tick direction
plt.rcParams['ytick.direction'] = 'in'  # Y tick direction
plt.rcParams['xtick.top'] = True  # Show ticks on top
plt.rcParams['ytick.right'] = True

## Signal Generation

In [ ]:
SIGNAL = None
@widgets.interact(
    fs = (125000, int(125e6), 1000),
    f_sig=(1000, int(200e3), 1000),
    p_sig=(0, 360, 10),
    periods = (10, 1000, 10),
    noise=(0, 1, 0.1),
    plot_period = (10, 100, 10),
    write_file = [False, True], 
    write_samples = (100, 100000, 100),
)
def plot_LIA(
    fs=12500e3,
    f_sig=125e3,
    p_sig=0,
    periods=1000,
    noise=0.1,
    plot_period=10,
    write_file=False,
    write_samples=2000,
):
    # Calculate samples needed for given frequency, periods and sampling rate
    samples_per_period = int(fs / f_sig)
    samples = samples_per_period * periods

    # Generate the time vector
    t = np.arange(samples) / fs

    # Generate the sine signal
    sig = np.sin(2 * np.pi * f_sig * t + np.deg2rad(p_sig))
    sig_noise = sig + np.random.normal(0, noise, samples)
    
    sin_ref = np.sin(2 * np.pi * f_sig * t)
    cos_ref = np.cos(2 * np.pi * f_sig * t)
    
    # Create global dict for signal
    global SIGNAL
    SIGNAL = {
        "sig": sig,
        "sig_noise": sig_noise,
        "samples": samples,
        "samples_per_period": samples_per_period,
        "f_sig": f_sig,
        "fs" : fs,
        "sin_ref": sin_ref,
        "cos_ref": cos_ref
    }
    
    # Create test_data directory
    if not os.path.exists('../sim/test_data'):
        os.makedirs('../sim/test_data')
        
    # Save signal to tv file
    if write_file:
        with open("../sim/test_data/i_data.tv", "w") as f:
            mix_sig = sig_noise * sin_ref
            for i in range(write_samples):
                s = int(mix_sig[i] * (2**16)-1)
                # convert to two's complement
                if s < 0:
                    s = (1 << 32) + s
                # convert to hex
                f.write(f"{s:08X}\n")
                
        with open("../sim/test_data/q_data.tv", "w") as f:
            mix_sig = sig_noise * cos_ref
            for i in range(write_samples):
                s = int(mix_sig[i] * (2**16)-1)
                # convert to two's complement
                if s < 0:
                    s = (1 << 32) + s
                # convert to hex
                f.write(f"{s:08X}\n")
                
    plt.figure(figsize=(15, 3))
    plt.plot(sig_noise, label="Signal + Noise")
    plt.plot(sig, label="Signal")
    plt.legend()
    plt.xlim(0, plot_period * samples_per_period)
    plt.show()
    
    print(f"Number of samples: {samples}")
    print(f"Number of samples per period: {samples_per_period}")
    print(f"Sampling frequency: {fs}")
    


## Filtering

## Filter Simulation with preconfigured filter coefficients

In [ ]:
####################################################################################################
class fir_filter:
    def __init__(self):
        self.coeffs = None
        self.info = None
###################################################################################################

# Global variable to store the FIR filter
FIR = None

# Get parameters from SIGNAL
sig = SIGNAL["sig"]
sig_noise = SIGNAL["sig_noise"]
samples = SIGNAL["samples"]
samples_per_period = SIGNAL["samples_per_period"]
f_sig = SIGNAL["f_sig"]
fs = SIGNAL["fs"]
sin_ref = SIGNAL["sin_ref"]
cos_ref = SIGNAL["cos_ref"]

@widgets.interact(
    fc=(0, 200000, 500),
    order=(50, 1000, 50),
    ref_scaling=(0, 16, 1),
    window=['hamming', 'blackman'],
    quantization=[True, False],
    quan_bits=(10, 25, 1),
    plot_periods=(10, 1000, 10)
    
)
def plot_LIA(
    fc=0.01*(fs/2),                 # [Hz] Cutoff frequency of the filter
    order=450,                      # Number of filter taps
    ref_scaling=14,                 # Scaling factor for the reference signal
    window='hamming',               # Window function for filter design
    quantization=True,              # Use quantized coefficients for filter
    quan_bits=24,                   # Number of bits for quantization, 24 bits is the default value
                                    # The coefficient width is 25, but the integer part 24 bits
    plot_periods=20                 # Number of periods to plot
    
):
    # Initialize the FIR filter
    global FIR
    FIR = fir_filter()
    
    # Mix the signals with the reference signal, all signals are downsampled
    I = sig_noise * (sin_ref * (2**ref_scaling -1))
    Q = sig_noise * (cos_ref * (2**ref_scaling -1))

    # Making sure that the number of taps is odd
    if order % 2 == 0:
        taps = order + 1
    else:
        taps = order

    # Calculate the filter coefficients
    coeffs_original = calc_fir_coeffs(taps, fc, fs=fs, window=window)
    # Quantize the coefficients and dequantize them, to simulate the precision loss of quantization
    coeffs_quan_dequan = quantize_and_dequantize_coeffs(coeffs_original, quan_bits) 
    # Choose the coefficients to use
    coeffs = coeffs_quan_dequan if quantization else coeffs_original
    # Store the coefficients in the FIR filter object for next cell
    FIR.coeffs = coeffs
    
    # Filter signals
    I_filt = lfilter(coeffs, 1, I) 
    Q_filt = lfilter(coeffs, 1, Q)

    # Calculate the phase and amplitude of the signal
    r, phi = calc_r_phi(I_filt, Q_filt)
    res = reconstruct_signal(r, phi, f_sig, fs, ref_scaling)
    
    # Calculate the frequency response of the filter
    w, h = freqz(coeffs_original, 1)
    w_q, h_q = freqz(coeffs_quan_dequan, 1)
    
    # Find the passband edge (0.99 gain) and stopband edge (0.01 gain)
    freq = w/(2 * np.pi) * fs#np.pi
    gain = np.abs(h)
    pb_edge = freq[np.where(gain <= 0.99)[0][0]]
    sb_edge = freq[np.where(gain <= 0.01)[0][0]] 
    f_cutoff = freq[np.where(gain <= 0.707)[0][0]] 
    
    # Print filter information
    FIR.info = f'order: {taps-1}, fs: {fs} Hz, fc: {fc} Hz, fc_norm: {fc/(fs/2)}, pb_edge: {pb_edge}, sb_edge: {sb_edge} | f_lock_in: {f_sig} Hz'
    print(FIR.info)
    
    ################################################################################################
    # Plot filter coefficients and frequency response
    ################################################################################################
    
    plt.figure(figsize=(20, 4))
    plt.subplot(1, 3, 1)
    plt.plot(coeffs, color='black', marker='.', label='Filter Coefficients')
    plt.title('FIR filter coefficients')
    plt.xlabel('coeffs(n)')
    plt.ylabel('Amplitude')
    plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot((w/np.pi), 20 * np.log10(abs(h)), color='black', label='Floating Point Coefficients')
    plt.plot((w_q/np.pi), 20 * np.log10(abs(h_q)), color='red', label='Fix Point Coefficients')
    plt.title('FIR filter frequency response')
    plt.xlabel(r'Normalized frequency (x$\pi$rad/sample)')
    plt.ylabel('Magnitude (dB)')
    plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(freq, gain, color='blue', label='Frequency Response')
    plt.axvline(pb_edge, c='k', linestyle='--', label=f'Passband Edge at {int(pb_edge)} Hz')
    plt.axvline(sb_edge, c='k', linestyle='--', label=f'Stopband Edge at {int(sb_edge)} Hz')
    plt.axvline(f_cutoff, c='red', linestyle='--',label=f'-3 dB Cutoff at {int(fc)} Hz')
    plt.axvspan(pb_edge, sb_edge, color='yellow', alpha=0.3, label=f'Transition Band Width: {int(sb_edge - pb_edge)} Hz')
    plt.yticks(np.arange(0, 1.1, 0.1))
    plt.xlim(0, 2*sb_edge)
    plt.title('FIR filter frequency response')
    plt.xlabel(r'Frequency (Hz)')
    plt.ylabel('Gain')
    plt.legend()
    
    ################################################################################################
    # Plot the signals & filtered signals and the reconstructed signal
    ################################################################################################
    plt_len = plot_periods * samples_per_period

    plt.figure(figsize=(20, 4))
    plt.subplot(1, 3, 1)
    plt.plot(I, label='In-phase')
    plt.plot(I_filt, label='In-phase filtered')
    plt.xlim(0, plt_len)
    plt.xlabel('Sample (n)')
    plt.ylabel('Counts')
    plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot(Q, label='Quadrature')
    plt.plot(Q_filt, label='Quadrature filtered')    
    plt.xlim(0, plt_len)
    plt.xlabel('Sample (n)')
    plt.ylabel('Counts')
    plt.legend()
        
    plt.subplot(1, 3, 3)
    plt.plot(sig, label='Signal of interest')
    plt.plot(res, label='Reconstructed signal')    
    plt.xlim(0, plt_len)
    plt.xlabel('Sample (n)')
    plt.ylabel('Voltage (V)')
    plt.legend()

    plt.show()

## Save Filter Coefficients to file with Xilinx readable .coe format

In [ ]:
################################################################################
# Save the FIR coefficients to a file in COE format with radix 2 (binary)
################################################################################
# Use the coefficients from the FIR filter design cell above and quantize them
coeffs = FIR.coeffs
# Number of bits for the coefficients are 25 bits but the integer part is 1 bit less
# These values are hardcoded in the hardware and should only be changed if the VHDL code is altered
COEFF_WIDTH = 25
QUAN_WIDTH = COEFF_WIDTH - 1

# Convert the coefficients from float to fixed-point integer 
coeffs = quantize_coeffs(coeffs, QUAN_WIDTH).astype(int)

# Open a file in write mode
with open('fir.coe', 'w') as file:
    # Write the header
    file.write(f'; File with {len(coeffs)} coefficients, each {COEFF_WIDTH} bits\n')
    file.write(f'; {FIR.info}\n')
    file.write('radix=2;\n')
    file.write('coefdata=\n')
    
    # Write the coefficients
    for i, c in enumerate(coeffs):
        # Convert integer to two's complement
        if c < 0:
            c = (1 << COEFF_WIDTH) + c
        # Convert coefficient to binary
        line = f'{c:0{COEFF_WIDTH}b}'
        # Commas in all lines except the last one
        if i < len(coeffs) - 1:
            line += ',\n'
        else:
            line += '\n'
        file.write(line)
